# Set up once, then run any capsule

*What do we need before any capsule notebook will run?*

Every capsule talks to models in a **Microsoft Foundry project**. This notebook gets us there once: sign in, create or reuse a project, write a `.env` file the capsules read, and confirm it works. After that, each capsule only checks for the few extra variables it needs.

```mermaid
flowchart LR
    A[Sign in to Azure] --> B[Foundry project]
    B --> C[.env<br/>endpoint + key]
    C --> D[Verify]
    D --> E[Open a capsule]
```

## 1. Before You Begin

- An **Azure subscription** with access to Microsoft Foundry ([sign in](https://ai.azure.com)).
- **This repo open in GitHub Codespaces or the devcontainer** — Python, the Azure CLI, and `requirements-dev.txt` are already installed. Running locally works too with Python 3.10+ and the [Azure CLI](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli).
- **Signed in to Azure** — run `az login` in a terminal (use `az login --use-device-code` in Codespaces).

## 2. Confirm the Azure sign-in

The next cell checks that the Azure CLI has an active account. If it fails, run `az login` in a terminal and re-run the cell.

In [ ]:
import json
import subprocess

result = subprocess.run(["az", "account", "show", "--output", "json"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("Not signed in. Run `az login` in a terminal, then re-run this cell.")
account = json.loads(result.stdout)
print(f"Signed in to subscription: {account['name']}")

## 3. Choose a Foundry project

We can reuse an existing project or create a new one.

- **Reuse** — in [ai.azure.com](https://ai.azure.com), open the project and note its **resource group** and **project name**.
- **Create** — pick a resource group, a project name, and a region. The next cell creates both if they don't exist. Full walkthrough: [Create a project in the Foundry portal](https://learn.microsoft.com/en-us/azure/foundry/how-to/create-projects).

Fill in the values below.

In [ ]:
MODE = "use"              # "use" an existing project, or "create" a new one
RESOURCE_GROUP = ""       # e.g. "rg-model-releases"
PROJECT_NAME = ""         # e.g. "model-releases"
LOCATION = "eastus2"      # only used when MODE == "create"

## 4. Write the `.env` file

The next cell runs [`scripts/setenv.sh`](scripts/setenv.sh). It copies [`scripts/sample.env`](scripts/sample.env) to `.env` at the repo root and fills in the project endpoint and API key from Azure. It won't overwrite an existing `.env` — delete it first, or run the script with `--force`, to start over. The `.env` file is git-ignored; never commit it.

In [ ]:
if MODE not in ("use", "create"):
    raise ValueError('MODE must be "use" or "create".')
if not RESOURCE_GROUP or not PROJECT_NAME:
    raise ValueError("Set RESOURCE_GROUP and PROJECT_NAME in the cell above.")

args = ["bash", "scripts/setenv.sh", f"--{MODE}", RESOURCE_GROUP, PROJECT_NAME]
if MODE == "create":
    args.append(LOCATION)
subprocess.run(args, check=True)

## 5. Verify your environment

Every capsule opens with a check like this one. When it prints `Ready`, any capsule's notebook can find our project.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)
required = ["MICROSOFT_FOUNDRY_ENDPOINT", "MICROSOFT_FOUNDRY_API_KEY"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(f"Missing {missing} in .env. Re-run section 4 or fill them in by hand.")
print("Ready:", os.environ["MICROSOFT_FOUNDRY_ENDPOINT"])

## 6. Deploy the models a capsule needs

Each capsule's first cell lists the deployments and extra variables it needs. To deploy a model:

1. In the project, open **Model catalog**, select the model, then **Deploy**.
2. Use the deployment name the capsule expects (the defaults in `.env` match the model name), or update `.env` to match yours.

Some models, such as MAI-Transcribe and MAI-Voice, need no deployment — the capsule names the model in the request. Full reference: [Deploy models in Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/how-to/deploy-models-openai).

## 7. Troubleshooting

- **`401` / `403`** — the API key is wrong, or our identity lacks the *Cognitive Services User* role. See [Foundry RBAC](https://learn.microsoft.com/en-us/azure/foundry/concepts/rbac-azure-ai-foundry).
- **`404` model not found** — the deployment name in `.env` doesn't match the portal. Names are case-sensitive.
- **Quota or region errors** — the model isn't available in our region, or quota is exhausted. See [Manage quotas](https://learn.microsoft.com/en-us/azure/foundry/how-to/quota).

Still stuck? [Open an issue](https://github.com/microsoft-foundry/model-releases/issues/new) with the failing cell's output.

## 8. Next steps

Pick a capsule from the [README](README.md) or the [CHANGELOG](CHANGELOG.md), open its notebook, and run it top to bottom.

## References

- [Quickstart: Get started with Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code)
- [Create a project in the Foundry portal](https://learn.microsoft.com/en-us/azure/foundry/how-to/create-projects)
- [Microsoft Foundry Models overview](https://learn.microsoft.com/en-us/azure/foundry/concepts/foundry-models-overview)